# Exploratory Analysis — Inventory Intelligence Dashboard

Three things this notebook documents:
1. The hourly transaction distribution that justifies a 14-hour trading window
2. A single-SKU proof of concept before scaling the model to the full catalogue
3. The velocity formula used to build `aggregated_catalog.csv` and how the curated SKUs were selected

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import poisson

df = pd.read_excel("../data/raw/Online Retail.xlsx")

# Basic cleaning — remove duplicates, missing customers, and invalid transactions
df = df.drop_duplicates()
df = df.dropna(subset=["CustomerID"])
df = df[df["Quantity"] > 0]
df = df[df["UnitPrice"] > 0]
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
df["CustomerID"] = df["CustomerID"].astype(int).astype(str)

df.info()

## 1. Trading Hours

Rather than assuming a trading window, the hour was extracted from each invoice date and transactions were counted per hour. The output below confirms where activity actually occurs.

In [4]:
df["Hour"] = df["InvoiceDate"].dt.hour
hourly_transactions = df["Hour"].value_counts().sort_index()
print(hourly_transactions)

Hour
6         1
7       379
8      8687
9     21927
10    37773
11    48365
12    70938
13    63019
14    53251
15    44790
16    23715
17    12941
18     2895
19     3233
20      778
Name: count, dtype: int64


Transactions peak at 12:00 (70,938) and are negligible before 06:00 and after 20:00. This confirms a 14-hour window (06:00–20:00) rather than a 24-hour assumption. Including the off-hours would artificially dilute the velocity for every product in the catalogue.

## 2. Single-SKU Proof of Concept

Before scaling to the full catalogue, the model was tested on one well-known product to confirm the logic worked as expected.

In [ ]:
df_uk = df[df["Country"] == "United Kingdom"].copy()

target_sku = "85123A"  # WHITE HANGING HEART T-LIGHT HOLDER
product_df = df_uk[df_uk["StockCode"] == target_sku].copy()

# Filter to trading hours only
product_df = product_df[
    (product_df["InvoiceDate"].dt.hour >= 6) &
    (product_df["InvoiceDate"].dt.hour < 20)
]

total_trading_days = product_df["InvoiceDate"].dt.date.nunique()
total_operational_hours = total_trading_days * 14
velocity = product_df["Quantity"].sum() / total_operational_hours

# Simulate a 6-hour sales gap
hours_since_last_sale = 6
expected_sales = velocity * hours_since_last_sale
probability_of_zero = poisson.pmf(0, expected_sales)
anomaly_confidence = (1 - probability_of_zero) * 100

print(f"SKU: {target_sku} — {product_df['Description'].iloc[0]}")
print(f"Sales Velocity      : {velocity:.2f} units/hour")
print(f"Expected in 6 hours : {expected_sales:.2f} units")
print(f"Anomaly Confidence  : {anomaly_confidence:.1f}%")

## 3. Full Catalogue Velocity Formula

The same logic scaled to all UK SKUs. Velocity = Total Units Sold ÷ Total Operational Hours, restricted to the confirmed 14-hour trading window. A floor of 0.2 units/hour was applied to prevent slow-moving products from requiring weeks before triggering any alert.

In [ ]:
df_trading = df_uk[
    (df_uk["InvoiceDate"].dt.hour >= 6) &
    (df_uk["InvoiceDate"].dt.hour < 20)
].copy()

total_trading_days = df_trading["InvoiceDate"].dt.date.nunique()
total_operational_hours = total_trading_days * 14

catalog_df = (
    df_trading.groupby("StockCode")
    .agg(
        Description=("Description", "first"),
        Total_Units=("Quantity", "sum")
    )
)

raw_velocity = catalog_df["Total_Units"] / total_operational_hours
catalog_df["Calculated_Velocity"] = np.maximum(0.2, raw_velocity)
catalog_df = catalog_df.sort_values(by="Calculated_Velocity", ascending=False).dropna()

print(f"{len(catalog_df)} unique SKUs in catalogue")
print(f"Total operational hours: {total_operational_hours}")

## 4. Curated SKU Selection

Six SKUs were selected by velocity band to cover all three alert tiers at the default dashboard settings (3 hours without sales, 95% alert sensitivity). The goal was to show a first-time visitor the full range of outputs without having to scroll through 3,645 products.

In [ ]:
curated_skus = ["22439", "22046", "22713", "17003", "90062", "22670"]

curated = catalog_df[catalog_df.index.isin(curated_skus)][["Description", "Calculated_Velocity"]].copy()

# Show what each SKU scores at default settings: 3 hours, 95% sensitivity
hours = 3
threshold = 95

def get_status(velocity, hours, threshold):
    lam = velocity * hours
    confidence = (1 - poisson.pmf(0, lam)) * 100
    if confidence >= threshold:
        status = "Critical"
    elif confidence >= threshold - 15:
        status = "Warning"
    else:
        status = "Normal"
    return round(confidence, 1), status

curated[["Anomaly Confidence", "Status"]] = curated["Calculated_Velocity"].apply(
    lambda v: pd.Series(get_status(v, hours, threshold))
)

print(curated.to_string())